# ML - Final Project
## Data Cleaning and Feature Extraction
**Project:** Multilingual Language Identification System

**Group Members:** 
1. Seyyed Sina Alizadeh Tabatabai - 810101477
2. Majid Sadeghinejad - 810101459
3. Matin Shiasi - 810101453

## 1. Introduction
The objective of this notebook is to prepare the raw audio dataset for the subsequent stages of classification and clustering. The dataset comprises audio clips from four languages: German, Italian, Korean, and Spanish, organized by gender (Male/Female).

Effective speech processing requires converting raw audio waveforms into compact, discriminative numerical representations. In this phase, we address three critical tasks:
1.  Data Cleaning: Removing non-informative segments (silence) to focus on the speech content.
2.  Data Augmentation: Injecting noise to increase dataset diversity and improve model robustness against real-world variations.
3.  Feature Extraction: transforming time-domain signals into spectral features (MFCCs, Chroma, Spectral Centroid) that capture linguistic characteristics.


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import warnings

warnings.filterwarnings('ignore')

DATASET_PATH = "../Data"
LANGUAGES = ["German", "Italian", "Korean", "Spanish"]
GENDERS = ["Male", "Female"]
SAMPLE_RATE = 22050

print("Libraries imported and configuration set.")

Libraries imported and configuration set.


## 2. Preprocessing: Data Cleaning (Silence Trimming)
Raw audio recordings often contain leading or trailing silence, or pauses between sentences. These silent segments do not carry linguistic information and can introduce bias or unnecessary dimensionality to the data.

**Methodology:**
We utilize librosa.effects.trim, which splits an audio signal into non-silent intervals based on a decibel threshold. By removing silence, we ensure that the extracted features represent active speech, thereby improving the signal-to-noise ratio for the classifier.


In [7]:
def load_and_trim_audio(file_path, sr=SAMPLE_RATE):
    try:
        y, _ = librosa.load(file_path, sr=sr)
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        return y_trimmed
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

## 3. Data Augmentation: Noise Injection
To prevent overfitting and simulate real-world recording conditions, we apply data augmentation. Since the dataset size is fixed, creating variations of the existing samples helps the model generalize better.

**Methodology:**
We implement Noise Injection by adding random Gaussian noise to the audio signal. This encourages the model to learn features that are robust to background interference rather than memorizing the clean training data.


In [8]:
def inject_noise(data, noise_factor=0.005):
    noise = np.random.randn(len(data))
    augmented_data = data + noise_factor * noise
    return augmented_data

## 4. Feature Extraction
Raw audio waveforms are high-dimensional and unsuitable for direct input into standard machine learning algorithms like SVMs or Random Forests. We extract specific features that characterize the timbre, pitch, and spectral content of the speech.

**Selected Features:**
1.  MFCCs (Mel-Frequency Cepstral Coefficients):
    *   Why: MFCCs approximate the human auditory system's response. They are the standard feature for speech recognition as they effectively capture the "timbre" or texture of the sound, which is crucial for distinguishing phonemes across languages.
    *   Stats: We extract the Mean and Variance of 13 MFCC coefficients to capture both the average spectral envelope and its temporal dynamics.

2.  Spectral Centroid:
    *   Why: Indicates the "center of mass" of the spectrum (perceived "brightness" of a sound). Different languages may have different frequency distributions based on their vowel/consonant usage.
    *   Stats: Mean.

3.  Chroma Feature:
    *   Why: Projects the spectrum onto 12 pitch classes. While more common in music, in speech, it can help capture tonal characteristics (important for tonal languages or specific intonation patterns).
    *   Stats: Mean.


In [ ]:
def extract_features(y, sr=SAMPLE_RATE):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc.T, axis=0)
    mfcc_var = np.var(mfcc.T, axis=0)
    
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spec_centroid)
    
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma.T, axis=0)
    
    
    features = np.hstack([mfcc_mean, mfcc_var, centroid_mean, chroma_mean])
    return features


## 5. Processing Pipeline
We iterate through the dataset structure (ML Dataset -> Language -> Gender), processing each file. 

**Pipeline Steps for each file:**
1.  Load and Trim Silence.
2.  Extract features from the **Original** audio.
3.  Generate an **Augmented** version (Noise Injection).
4.  Extract features from the **Augmented** audio.
5.  Label data with Language and Gender.


In [10]:
data_records = []

print("Starting Feature Extraction Process...")

for language in LANGUAGES:
    for gender in GENDERS:
        folder_path = os.path.join(DATASET_PATH, language, gender)
        
        if not os.path.exists(folder_path):
            print(f"Directory not found - {folder_path}")
            continue
            
        print(f"Processing: {language} - {gender}")
        
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            
            if not filename.lower().endswith(('.mp3', '.mp4')):
                print("Audio format isn't .mp3")
                continue
                
            y_trimmed = load_and_trim_audio(file_path)
            
            if y_trimmed is None or len(y_trimmed) == 0:
                print("Audio is just silence.")
                continue
            
            features_original = extract_features(y_trimmed)
            
            record_original = {
                'filename': filename,
                'label': language,
                'gender': gender,
                'type': 'original'
            }
            
            for i, feat in enumerate(features_original):
                record_original[f'feature_{i}'] = feat
            data_records.append(record_original)
            

            y_noise = inject_noise(y_trimmed)
            features_augmented = extract_features(y_noise)
            
            record_aug = {
                'filename': f"noise_{filename}",
                'label': language,
                'gender': gender,
                'type': 'augmented'
            }
            for i, feat in enumerate(features_augmented):
                record_aug[f'feature_{i}'] = feat
            data_records.append(record_aug)

print(f"\nProcessing Complete. Total samples processed: {len(data_records)}")


Starting Feature Extraction Process...
Processing: German - Male
Processing: German - Female
Processing: Italian - Male
Processing: Italian - Female
Processing: Korean - Male
Processing: Korean - Female
Processing: Spanish - Male
Processing: Spanish - Female

Processing Complete. Total samples processed: 1440


## Feature Description and Mapping

In this project, we extract a total of **39 numerical features** per audio sample to represent its linguistic characteristics. These features are flattened into a 1D vector and stored in columns feature_0 through feature_38.

The features are concatenated in the following order: MFCC Means, MFCC Variances, Spectral Centroid, Chroma Means.

| Feature Group | Count | Column Indices | Description |
| :--- | :--- | :--- | :--- |
| MFCC (Mean) | 13 | feature_0 - feature_12 | Captures the average timbre and phonetic content. Corresponds to the static spectral envelope of the speech. |
| MFCC (Variance) | 13 | feature_13 - feature_25 | Captures the temporal dynamics and variability of the speech. High variance often indicates rapid articulation or significant acoustic changes. |
| Spectral Centroid | 1 | feature_26 | Represents the "center of mass" of the spectrum, correlating with the perceived brightness of the sound (e.g., high-frequency consonants vs. low-frequency vowels). |
| Chroma | 12 | feature_27 - feature_38 | Projects the spectrum onto 12 pitch classes. Useful for capturing tonal characteristics and intonation patterns specific to certain languages. |

Total Feature Vector Length: $13 + 13 + 1 + 12 = 39$ features.


## 6. Saving Data
The extracted features are aggregated into a Pandas DataFrame and saved as a CSV file. This CSV will serve as the input for the Classification and Clustering notebooks.


In [14]:
df = pd.DataFrame(data_records)

print(f"DataFrame Shape: {df.shape}")
print("Class Distribution:")
print(df['label'].value_counts())

output_filename = '../Output/CSV/extracted_features.csv'
df.to_csv(output_filename, index=False)

print(f"\nFeatures saved successfully to {output_filename}")
df.head()


DataFrame Shape: (1440, 43)
Class Distribution:
label
German     360
Italian    360
Korean     360
Spanish    360
Name: count, dtype: int64

Features saved successfully to ../Output/CSV/extracted_features.csv


,filename,label,gender,type,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,...,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38
0,810100147_male_german_voice18.mp3,German,Male,original,-360.147736,81.419189,6.685035,37.361408,10.992982,14.815589,...,0.392517,0.359050,0.355582,0.331569,0.322169,0.378176,0.433062,0.460627,0.505175,0.567532
1,noise_810100147_male_german_voice18.mp3,German,Male,augmented,-243.265922,44.717608,14.929080,18.284694,9.742257,7.490328,...,0.486399,0.444799,0.444285,0.404756,0.379714,0.440925,0.495623,0.516760,0.548733,0.619064
2,810103040_male_german_voice9.mp3,German,Male,original,-415.792633,92.994125,11.725086,37.884670,8.752963,13.205075,...,0.239160,0.284149,0.361868,0.362675,0.364930,0.433077,0.509956,0.542328,0.554988,0.492306
3,noise_810103040_male_german_voice9.mp3,German,Male,augmented,-263.376941,40.922892,19.084346,14.412500,7.726391,5.725642,...,0.399446,0.411116,0.462347,0.431228,0.426357,0.490274,0.572535,0.624915,0.656147,0.604483
4,810101399_male_german_voice02.mp3,German,Male,original,-350.339386,81.340599,5.093731,35.455444,8.609697,10.642005,...,0.306435,0.288379,0.328292,0.324297,0.308916,0.333570,0.422445,0.497449,0.508424,0.497756
